# Sport and recreation facilities: load and clean

This notebook loads the Victorian sport and recreation facilities
workbook, removes duplicate facilities, and finds the facilities that
have no coordinates so they can be geocoded from their addresses.

## Imports and configuration

The file path and the columns we care about are stored as upper-case
constants, following the PEP 8 convention for module-level constants.

In [17]:
import pandas as pd
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

FACILITIES_PATH = "../../external_data/sport_recreational_facilities.xlsx"
LOCATION_COLUMNS = ["Facility ID", "Latitude", "Longitude"]

USER_AGENT = "vic-facility-geocoding-nikhil-gaba"
VIC_LAT_RANGE = (-39.3, -33.9)
VIC_LON_RANGE = (140.9, 150.1)

GEOCODE_OUTPUT_PATH = "../../external_data/geocode_results.csv"
EXACT_FACILITIES_PATH = "../../external_data/sports_rec_facilities_exact.csv"

EXACT_LEVELS = ["original", "full_address", "facility_name"]

SWAPPED_COORDINATE_IDS = ["SWANHI5449", "SWANHI5447", "SWANHI5448"]
DECIMAL_SHIFT_DIVISORS = {"MURRIN5458": 1e6, "MURRIN5461": 1e5}
MANUAL_LATITUDES = {"BENDIG5468": -36.7148}

## Inspect the workbook

Open the workbook once and list its sheet names, so we can confirm
that sheet 1 is the one we want before reading it.

In [2]:
excel_file = pd.ExcelFile(FACILITIES_PATH)
print(excel_file.sheet_names)

['wholeIFMD', 'Alphabetised by facilityname', 'count if 6', 'Sheet1']


## Read sheet 1

Read the first sheet (index 0) into a DataFrame. `facilities_raw` is
never modified; every cleaning step below produces a new DataFrame.

In [3]:
facilities_raw = pd.read_excel(excel_file, sheet_name=0)

## Check for duplicate facilities

`Facility ID` should be unique. If there are duplicate facility ID's, they will be dropped.

In [4]:
n_duplicate_ids = facilities_raw["Facility ID"].duplicated().sum()
print(f"Repeated Facility IDs: {n_duplicate_ids}")

Repeated Facility IDs: 4590


## Deduplicate

Keep the first row for each `Facility ID`. The original index is
preserved, so every remaining row can still be traced back to its
position in the Excel sheet.

In [5]:
facilities = facilities_raw.drop_duplicates(subset="Facility ID")

print(f"Rows before: {len(facilities_raw)}")
print(f"Rows after: {len(facilities)}")

Rows before: 9590
Rows after: 5000


## Check the location columns for nulls

Keep only the facility ID and the coordinate columns, then count the
nulls in each column and list the rows where any of the three is
missing.

In [6]:
facility_locations = facilities[LOCATION_COLUMNS].copy()

print(facility_locations.isna().sum())

rows_missing_location = facility_locations[
    facility_locations.isna().any(axis=1)
]
print(
    f"{len(rows_missing_location)} of {len(facility_locations)} "
    "rows have a null"
)
rows_missing_location.head(20)

Facility ID      0
Latitude       319
Longitude      319
dtype: int64
319 of 5000 rows have a null


,Facility ID,Latitude,Longitude
39,ALPINE11452,NaN,NaN
40,ALPINE11495,NaN,NaN
622,BAWBAW12139,NaN,NaN
671,BAWBAW12086,NaN,NaN
672,BAWBAW12134,NaN,NaN
673,BAWBAW12138,NaN,NaN
680,BAWBAW11521,NaN,NaN
683,BAWBAW12133,NaN,NaN
693,BAWBAW12140,NaN,NaN
832,BAYSID12380,NaN,NaN


## Facilities to geocode

Select the full rows (all columns, not just the location columns)
where the latitude or longitude is missing. These are the facilities
we will geocode from their addresses. 

In [7]:
missing_coordinates = facilities[
    facilities["Latitude"].isna() | facilities["Longitude"].isna()
].copy()

missing_coordinates.head(10)

,Facility ID,LGA Name,Facility Name,Facility_AutoNumber,Street #,Street Name,Street Type,Suburb/Town,Pcode,Melway Ref,...,Number of Field/Courts,Field/Surface Type,Age of Facility,Condition of Facility,Facility Upgrade Age,Changerooms,Facility Features,Spectator numbers for seating/shelter,FullAddress,MelwaysVicRoadsRef
39,ALPINE11452,Alpine Shire Council,Bright Community Stadium,5131,NaN,Deacon,Avenue,BRIGHT,3741.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Deacon Avenue, BRIGHT 3741",NaN
40,ALPINE11495,Alpine Shire Council,Mt Beauty Community Stadium,5132,NaN,Pool,Road,MOUNT BEAUTY,3699.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Pool Road, MOUNT BEAUTY 3699",NaN
622,BAWBAW12139,Baw Baw Shire Council,Superfit Boxing,372,19,Albert,Street,WARRAGUL,3820.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"19 Albert Street, WARRAGUL 3820",704 F6
671,BAWBAW12086,Baw Baw Shire Council,Gregory's Fitness Centre,319,435,Main Nerrim,Road,DROUIN WEST,3818.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"435 Main Nerrim Road, DROUIN WEST 3818",96 F3
672,BAWBAW12134,Baw Baw Shire Council,Jada Fitness,367,NaN,Pearse,Street,WARRAGUL,3820.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Pearse Street, WARRAGUL 3820",704 H7
673,BAWBAW12138,Baw Baw Shire Council,Studio Samsara Fitness,371,7,Olympic,Street,WARRAGUL,3820.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"7 Olympic Street, WARRAGUL 3820",704 H7
680,BAWBAW11521,Baw Baw Shire Council,Warragul Gymnastic Club,5136,40,Alfred,Street,WARRAGUL,3820.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"40 Alfred Street, WARRAGUL 3820",NaN
683,BAWBAW12133,Baw Baw Shire Council,Goju Martial Arts,366,NaN,Burke,Street,WARRAGUL,3820.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Burke Street, WARRAGUL 3820",704 A5
693,BAWBAW12140,Baw Baw Shire Council,Extinction Mixed Martial Arts,373,133,North,Road,WARRAGUL,3820.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"133 North Road, WARRAGUL 3820",704 J7
832,BAYSID12380,Bayside City Council,Gecko Kids Fitness,442,338,South,Road,HAMPTON EAST,3188.0,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"338 South Road, HAMPTON EAST 3188",NaN


## Geocode facilities with missing coordinates

`missing_coordinates` holds the facilities with no latitude or
longitude. We look them up with Nominatim, the free OpenStreetMap
geocoder, using the address columns in the workbook.

The code is split into small functions so each step can be read and
tested on its own.

### Text-cleaning helpers

Excel data is messy: blanks arrive as `NaN`, postcodes arrive as floats
(`3741.0`) and suburbs are upper case (`BRIGHT`). These helpers turn
each value into a clean string, or `None` if there is nothing usable.

In [8]:
def clean_text(value):
    """Return a stripped string, or None if the value is blank or NaN."""
    if pd.isna(value) or str(value).strip() == "":
        return None
    return str(value).strip()


def format_postcode(value):
    """Convert a postcode such as 3741.0 to the string '3741'."""
    text = clean_text(value)
    if text is None:
        return None
    try:
        return str(int(float(text)))
    except ValueError:
        return None


def get_suburb(row):
    """Return the suburb in title case (BRIGHT -> Bright), or None."""
    suburb = clean_text(row["Suburb/Town"])
    return suburb.title() if suburb else None


def build_street(row):
    """Join street name and type, e.g. 'Main Nerrim' and 'Road'."""
    parts = [
        clean_text(row["Street Name"]),
        clean_text(row["Street Type"]),
    ]
    return " ".join(part for part in parts if part) or None

### Building the queries

Each facility can be searched in up to four ways, from most to least
precise. `build_query_tiers` returns them in that order, skipping any
tier the facility has no data for:

| Tier | Query | Typical accuracy |
|---|---|---|
| `full_address` | street number + street + suburb + postcode | building |
| `facility_name` | facility name + suburb | good if mapped in OSM |
| `street_only` | street + suburb + postcode | somewhere on the street |
| `suburb_only` | suburb + postcode | suburb centre |

Address-based tiers use Nominatim's *structured* query (separate
street, city and postcode fields), which is more reliable than one
free-text string. Every query is restricted to Victoria, Australia.

In [9]:
def build_structured_query(street=None, suburb=None, postcode=None):
    """Build a Nominatim structured query restricted to Victoria."""
    query = {"state": "Victoria", "country": "Australia"}
    if street:
        query["street"] = street
    if suburb:
        query["city"] = suburb
    if postcode:
        query["postalcode"] = postcode
    return query


def build_query_tiers(row):
    """Return (level, query) pairs ordered from most to least precise."""
    street_number = clean_text(row["Street #"])
    street = build_street(row)
    suburb = get_suburb(row)
    postcode = format_postcode(row["Pcode"])
    facility_name = clean_text(row["Facility Name"])

    tiers = []
    if street_number and street:
        full_street = f"{street_number} {street}"
        query = build_structured_query(full_street, suburb, postcode)
        tiers.append(("full_address", query))
    if facility_name and suburb:
        query = f"{facility_name}, {suburb}, Victoria, Australia"
        tiers.append(("facility_name", query))
    if street:
        query = build_structured_query(street, suburb, postcode)
        tiers.append(("street_only", query))
    if suburb:
        query = build_structured_query(None, suburb, postcode)
        tiers.append(("suburb_only", query))
    return tiers

### Validating a match and geocoding one facility

A geocoder can return a confident answer that is simply wrong, such as
a street with the same name in another suburb. `is_plausible_match`
rejects any result that:

- falls outside Victoria's bounding box, or
- has an address containing neither the expected suburb nor the
  expected postcode.

`geocode_facility` tries each tier in turn and returns the first
plausible match, together with the tier that produced it
(`geocode_level`) and the address the geocoder matched
(`matched_as`), so results can be audited later. If every tier fails,
`geocode_level` is `"failed"` and the coordinates are empty.

In [10]:
def is_plausible_match(location, suburb, postcode):
    """Check a match is inside Victoria and near the expected place."""
    in_victoria = (
        VIC_LAT_RANGE[0] <= location.latitude <= VIC_LAT_RANGE[1]
        and VIC_LON_RANGE[0] <= location.longitude <= VIC_LON_RANGE[1]
    )
    address = location.address.lower()
    in_expected_place = (
        (suburb is not None and suburb.lower() in address)
        or (postcode is not None and postcode in address)
    )
    return in_victoria and in_expected_place


def geocode_facility(row, geocode):
    """Geocode one facility, trying progressively less precise queries.

    Returns a Series with the coordinates, the tier that matched (or
    'failed') and the address the geocoder matched.
    """
    suburb = get_suburb(row)
    postcode = format_postcode(row["Pcode"])

    for level, query in build_query_tiers(row):
        location = geocode(query, timeout=10)
        if location and is_plausible_match(location, suburb, postcode):
            return pd.Series({
                "Latitude": location.latitude,
                "Longitude": location.longitude,
                "geocode_level": level,
                "matched_as": location.address,
            })

    return pd.Series({
        "Latitude": None,
        "Longitude": None,
        "geocode_level": "failed",
        "matched_as": None,
    })

### Run the geocoder

`RateLimiter` wraps Nominatim so that calls are at least 1 seconds
apart, which respects its limit of one request per second. It also
retries transient errors and returns `None` instead of raising.

With 319 facilities and up to four queries each, this cell
takes between about 7 and 25 minutes. The results are saved to CSV
straight away so they survive a kernel restart.

In [11]:
geolocator = Nominatim(user_agent=USER_AGENT)
rate_limited_geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.01,
)

geocode_results = missing_coordinates.apply(
    geocode_facility,
    axis=1,
    geocode=rate_limited_geocode,
)
geocode_results.to_csv(GEOCODE_OUTPUT_PATH)

print(geocode_results["geocode_level"].value_counts())

geocode_level
full_address     212
street_only       47
suburb_only       41
facility_name     16
failed             3
Name: count, dtype: int64


### Merge the results back

Copy every geocoding result into a new DataFrame, `facilities_geocoded`,
aligning on the index. Every row starts with `geocode_level` set to
`"original"`, meaning its coordinates came from the source data.
Geocoded rows, including those that failed, are overwritten with their
tier, so a failed row is labelled `failed` and not `original`.

In [12]:
facilities_geocoded = facilities.copy()
facilities_geocoded["geocode_level"] = "original"
facilities_geocoded["matched_as"] = None

merge_columns = ["Latitude", "Longitude", "geocode_level", "matched_as"]
facilities_geocoded.loc[geocode_results.index, merge_columns] = (
    geocode_results[merge_columns]
)

### Check the outcome

First, count how many facilities landed at each accuracy level and how
many still have no coordinates. Then look at a random sample of
geocoded rows, comparing the address in the source data (`FullAddress`)
with the address the geocoder matched (`matched_as`). Look for wrong
suburbs and similarly named streets.

In [13]:
print(facilities_geocoded["geocode_level"].value_counts())
print(facilities_geocoded[["Latitude", "Longitude"]].isna().sum())

review = missing_coordinates[["Facility Name", "FullAddress"]].join(
    geocode_results
)
review.sample(n=20, random_state=42)

geocode_level
original         4681
full_address      212
street_only        47
suburb_only        41
facility_name      16
failed              3
Name: count, dtype: int64
Latitude     3
Longitude    3
dtype: int64


,Facility Name,FullAddress,Latitude,Longitude,geocode_level,matched_as
3338,Genesis,"320 High Street, BELMONT 3216",-38.187481,144.329085,full_address,"312-320, High Street, Belmont, Geelong, Victor..."
8590,Stratford Swimming Pool,"McMillian Street, STRATFORD 3862",-37.970193,147.078324,facility_name,"Stratford Swimming Pool, McMillan Street, Stra..."
1309,Strathcona Rythmic Gymnastic Club (Strathcona ...,"34 Scott Street, CANTERBURY 3126",-37.829800,145.079919,full_address,"34, Scott Street, Canterbury, Melbourne, Victo..."
8479,Bishop Phelan Stadium,"48 Pearson Street, SALE 3850",-38.107707,147.062600,full_address,"Pearson Street, Sale, Victoria, 3850, Australia"
832,Gecko Kids Fitness,"338 South Road, HAMPTON EAST 3188",-37.933211,145.032543,full_address,"336-338, South Road, Hampton East, Melbourne, ..."
3650,Portland Angling Club,"Lee Breakwater Road, PORTLAND 3305",-38.352604,141.611268,street_only,"Lee Breakwater Road, Portland, Victoria, 3305,..."
6780,Peninsula Gymnastic,"212 Boneo Road, ROSEBUD 3939",-38.381914,144.888378,full_address,"212, Boneo Road, Rosebud, Melbourne, Victoria,..."
7786,Hamilton and District Gym Club,"266 Mt Baimbridge Road, HAMILTON 3300",-37.720496,142.015601,full_address,"266, Mount Baimbridge Road, Hamilton, Victoria..."
3012,Not a Gym,"46 Peninsula Boulevard, SEAFORD 3198",-38.115872,145.139353,full_address,"42-46, Peninsula Boulevard, Seaford, Melbourne..."
4513,Self-ish Health Club,"150 Chesterville Road, CHELTENHAM 3192",-37.949860,145.059428,full_address,"150, Chesterville Road, Cheltenham, Melbourne,..."


### Keep only exact locations

Keep the facilities whose coordinates came from the source data
(`original`) or from an exact geocode (`full_address`, `facility_name`).
Facilities located only to a street or suburb (`street_only`,
`suburb_only`) or not located at all (`failed`) are dropped, because
their coordinates would put a facility in the wrong place without any
visible warning.

`facilities_exact` is the table used from here on.

In [14]:
is_exact = facilities_geocoded["geocode_level"].isin(EXACT_LEVELS)
facilities_exact = facilities_geocoded[is_exact].copy()

n_dropped = len(facilities_geocoded) - len(facilities_exact)
print(f"Kept {len(facilities_exact)} facilities, dropped {n_dropped}")
print(facilities_geocoded.loc[~is_exact, "geocode_level"].value_counts())

Kept 4909 facilities, dropped 91
geocode_level
street_only    47
suburb_only    41
failed          3
Name: count, dtype: int64


### Check the original coordinates

The geocoded rows were already checked against Victoria's bounding box
inside `is_plausible_match`. The `original` rows came straight from the
source data and were not, so count how many fall outside the box (for
example swapped or zeroed values). A result of 0 means no further
cleaning is needed.

In [19]:
is_original = facilities_exact["geocode_level"] == "original"
is_in_victoria = (
    facilities_exact["Latitude"].between(*VIC_LAT_RANGE)
    & facilities_exact["Longitude"].between(*VIC_LON_RANGE)
)

n_outside = (is_original & ~is_in_victoria).sum()
print(f"Original coordinates outside Victoria: {n_outside}")

Original coordinates outside Victoria: 0


### Inspect the out-of-range coordinates

List the `original` facilities whose coordinates fall outside Victoria.
Look for swapped latitude and longitude, positive latitudes or zeros,
which can often be corrected by hand.

In [16]:
outside_victoria = facilities_exact.loc[
    is_original & ~is_in_victoria,
    ["Facility ID", "Facility Name", "Suburb/Town", "Latitude", "Longitude"],
]
outside_victoria

,Facility ID,Facility Name,Suburb/Town,Latitude,Longitude
1129,BENDIG5468,McKern Skate Park,EAGLEHAWK,-3.971480e+01,144.254114
7092,MURRIN5458,Marysville Golf Club,MARYSVILLE,-3.749455e+07,145.747764
7127,MURRIN5461,Toolangi Tennis Club,TOOLANGI,-3.753826e+06,145.473367
8138,SWANHI5449,Woorinen Recreation Reserve,WOORINEN,1.434682e+02,-35.271184
8158,SWANHI5447,E F Butler Reserve,TYNTYNDER SOUTH,1.435285e+02,-35.282526
8202,SWANHI5448,Murraydale Tennis Club,NaN,1.435841e+02,-35.285388


### Correct known coordinate errors

Six `original` coordinates were outside Victoria. Each was checked by
hand and fits one of three patterns:

- **Swapped latitude and longitude:** the two columns are swapped back.
  `.values` is required so pandas swaps the data and does not realign
  it by column name, which would silently do nothing.
- **Lost decimal point:** the latitude is divided by the power of ten
  it was inflated by.
- **Mistyped digit:** the latitude is set to a value confirmed on a map.

The corrected rows are displayed at the end so they can be checked.

In [18]:
is_swapped = facilities_exact["Facility ID"].isin(SWAPPED_COORDINATE_IDS)
facilities_exact.loc[is_swapped, ["Latitude", "Longitude"]] = (
    facilities_exact.loc[is_swapped, ["Longitude", "Latitude"]].values
)

for facility_id, divisor in DECIMAL_SHIFT_DIVISORS.items():
    is_facility = facilities_exact["Facility ID"] == facility_id
    facilities_exact.loc[is_facility, "Latitude"] /= divisor

for facility_id, latitude in MANUAL_LATITUDES.items():
    is_facility = facilities_exact["Facility ID"] == facility_id
    facilities_exact.loc[is_facility, "Latitude"] = latitude

corrected_ids = (
    SWAPPED_COORDINATE_IDS
    + list(DECIMAL_SHIFT_DIVISORS)
    + list(MANUAL_LATITUDES)
)
is_corrected = facilities_exact["Facility ID"].isin(corrected_ids)
facilities_exact.loc[
    is_corrected,
    ["Facility ID", "Facility Name", "Suburb/Town", "Latitude", "Longitude"],
]

,Facility ID,Facility Name,Suburb/Town,Latitude,Longitude
1129,BENDIG5468,McKern Skate Park,EAGLEHAWK,-36.714800,144.254114
7092,MURRIN5458,Marysville Golf Club,MARYSVILLE,-37.494546,145.747764
7127,MURRIN5461,Toolangi Tennis Club,TOOLANGI,-37.538260,145.473367
8138,SWANHI5449,Woorinen Recreation Reserve,WOORINEN,-35.271184,143.468164
8158,SWANHI5447,E F Butler Reserve,TYNTYNDER SOUTH,-35.282526,143.528459
8202,SWANHI5448,Murraydale Tennis Club,NaN,-35.285388,143.584072


## Export the cleaned facilities

Save the facilities with exact coordinates to a CSV file, keeping only
the facility ID, latitude and longitude. The proximity calculation
will read this file in a separate notebook.

`index=False` stops pandas writing its row numbers as an extra column.
No rounding is applied, so the coordinates keep their full precision.

In [20]:
facilities_export = facilities_exact[LOCATION_COLUMNS]
facilities_export.to_csv(EXACT_FACILITIES_PATH, index=False)

print(f"Exported {len(facilities_export)} facilities")
print(facilities_export.isna().sum())

Exported 4909 facilities
Facility ID    0
Latitude       0
Longitude      0
dtype: int64
